# Scout de Jóvenes Promesas — Paso 4
## Integración LLM — Recomendaciones Automáticas de Scouting

**LLM utilizado:** Llama 3.3 70B vía Groq API (gratuito)

**Inputs:**
- `outputs/paso2/scouting_dataset_v2_con_predicciones.csv` → valor predicho
- `outputs/paso3/simulaciones_montecarlo.csv` → proyecciones futuras
- `outputs/paso3/joyas_ocultas_top50.csv` → oportunidades de mercado

**Objetivo:**
Convertir los outputs numéricos de los pasos anteriores en **reportes de scouting
profesionales en lenguaje natural**. El LLM recibe los datos estructurados de cada
jugador y genera análisis que un director deportivo puede leer y usar directamente
para tomar decisiones de fichaje.

**Qué puede hacer este sistema:**
- Reporte individual completo de cualquier jugador del dataset
- Comparativa head-to-head de candidatos para la misma posición
- Identificación y análisis de joyas ocultas (alto potencial, bajo valor actual)
- Consultas interactivas en tiempo real: `analizar_jugador("nombre")`

In [2]:
# Imports y configuración
# ===========================================================

import os
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from groq import Groq
from dotenv import load_dotenv

warnings.filterwarnings("ignore")

# ── Rutas
DATA_V1    = Path("../outputs/paso1/scouting_dataset_v1.csv")
DATA_V2    = Path("../outputs/paso2/scouting_dataset_v2_con_predicciones.csv")
SIMULACION = Path("../outputs/paso3/simulaciones_montecarlo.csv")
JOYAS      = Path("../outputs/paso3/joyas_ocultas_top50.csv")
OUTPUT_DIR = Path("../outputs/paso4")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

load_dotenv()
API_KEY = os.getenv("GROQ_API_KEY")

client = Groq(api_key=API_KEY)
MODELO = "llama-3.3-70b-versatile"

PALETTE = {
    "primary":   "#1D9E75",
    "secondary": "#7F77DD",
    "accent":    "#EF9F27",
    "danger":    "#D85A30",
    "neutral":   "#888780",
    "light":     "#F4F3EE",
    "dark":      "#2C2C2A",
}
plt.rcParams.update({
    "figure.facecolor": PALETTE["light"],
    "axes.facecolor":   PALETTE["light"],
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "font.size": 11,
})

def save_fig(fig, name):
    path = OUTPUT_DIR / f"{name}.png"
    fig.savefig(path, dpi=150, bbox_inches="tight", facecolor=PALETTE["light"])
    plt.close(fig)
    print(f"  ✓ {name}.png")

print(f"✓ Cliente Groq inicializado")
print(f"  Modelo: {MODELO}")

✓ Cliente Groq inicializado
  Modelo: llama-3.3-70b-versatile


## 1. Configuración — Groq API Key

**¿Qué es Groq?**
Groq es una plataforma que ofrece inferencia de LLMs de código abierto
(Llama, Mixtral, Gemma) con hardware especializado (LPU). La API es gratuita
con límites generosos: ~6,000 tokens/minuto y 500,000 tokens/día

**Cómo obtener la key (5 minutos, sin tarjeta):**
1. Ve a https://console.groq.com/keys
2. Crea una cuenta con Google o email
3. Click en "Create API Key"
4. Copia la key — empieza con `gsk_...`

**Cómo configurarla en PyCharm:**
```
crear archivo .nev > agregar api_key=api_key > en el codigo de configuracion agregar > agregar .env a .gitignore
    from dotenv import load_dotenv
    import os
    
    load_dotenv()

    api_key = os.getenv("GROQ_API_KEY")
```

**Por qué no escribirla directamente en el código:**
Al momento de subir el notebook a GitHub con la key escrita,
cualquier persona puede verla y usarla con tu cuenta.
La variable de entorno resuelve esto — el código la lee pero nunca la almacena.

**Modelo seleccionado: `llama-3.3-70b-versatile`**
Es el modelo de mayor calidad disponible gratuitamente en Groq.
Llama 3.3 de Meta tiene excelente capacidad de análisis en español
y manejo de datos estructurados, que es exactamente lo que necesitamos.

## 2. Arquitectura del sistema

```
player_id
    │
    ▼
construir_contexto_jugador()
    │  Lee df_base + df_sim
    │  Consolida 30+ variables en un dict limpio
    │  Calcula métricas derivadas
    ▼
prompt_reporte_individual(ctx)
    │  Formatea los datos como texto estructurado
    │  Incluye proyecciones Montecarlo si están disponibles
    │  Define las secciones del reporte esperado
    ▼
llamar_llm(prompt)
    │  Llama a Groq API (Llama 3.3 70B)
    │  System prompt: rol de analista de scouting
    │  Temperatura 0.3: analítico y consistente
    ▼
Reporte en Markdown
    │
    ├── guardar_reporte() → .md en outputs/paso4/
    └── mostrar en notebook
```

**Tres tipos de análisis disponibles:**

| Función | Uso | Cuándo usarlo |
|---|---|---|
| `prompt_reporte_individual` | Análisis profundo de 1 jugador | Jugador específico de interés |
| `prompt_comparativa` | Head-to-head de 2-4 jugadores | Decidir entre candidatos para la misma posición |
| `prompt_joyas_ocultas` | Top oportunidades de mercado | Sesiones de scouting proactivo |

In [6]:
# Carga de datos
# ===========================================================

if DATA_V2.exists():
    df_base   = pd.read_csv(DATA_V2, low_memory=False)
    valor_col = "pred_market_value_eur"
    print(f"✓ Dataset v2 (con predicciones): {df_base.shape}")
else:
    df_base   = pd.read_csv(DATA_V1, low_memory=False)
    valor_col = "latest_value"
    print(f"✓ Dataset v1: {df_base.shape}")
    print("  ⚠ Sin predicciones del modelo — usando latest_value")

df_base["nombre_limpio"] = df_base["player_name"].str.split("(").str[0].str.strip()

df_sim = pd.read_csv(SIMULACION) if SIMULACION.exists() else None
print(f"{'✓' if df_sim is not None else '⚠'} Simulaciones Montecarlo: {'cargadas' if df_sim is not None else 'no disponibles'}")

df_joyas = pd.read_csv(JOYAS) if JOYAS.exists() else None
print(f"{'✓' if df_joyas is not None else '⚠'} Joyas ocultas: {'cargadas' if df_joyas is not None else 'no disponibles'}")

print(f"\nJugadores con valor disponible: {df_base[valor_col].notna().sum():,}")

✓ Dataset v2 (con predicciones): (31405, 61)
✓ Simulaciones Montecarlo: cargadas
✓ Joyas ocultas: cargadas

Jugadores con valor disponible: 31,405


In [7]:
# Constructor de contexto del jugador
# ===========================================================

def construir_contexto_jugador(player_id: int) -> dict:
    """
    Extrae y estructura toda la información de un jugador
    en un diccionario limpio para enviarlo al LLM.
    """
    row = df_base[df_base["player_id"] == player_id]
    if row.empty:
        raise ValueError(f"player_id {player_id} no encontrado")
    row = row.iloc[0]

    def si(val, default=0):
        return int(val) if pd.notna(val) else default

    def sf(val):
        return float(val) if pd.notna(val) else None

    def ss(val):
        return str(val) if pd.notna(val) else "N/D"

    ctx = {
        "nombre":       row["nombre_limpio"],
        "edad":         round(float(row["age"]), 1),
        "posicion":     ss(row.get("main_position")),
        "posicion_det": ss(row.get("position")),
        "pais":         ss(row.get("country_of_birth", row.get("citizenship"))),
        "nacionalidad": ss(row.get("citizenship")),
        "pie":          ss(row.get("foot")),
        "altura_cm":    si(row.get("height")) if row.get("height", 0) > 0 else None,
        "es_eu":        bool(row.get("is_eu", False)),
        "club_actual":  ss(row.get("current_club_name")),
        "liga":         ss(row.get("club_league")),
        "pais_club":    ss(row.get("club_country")),
        "fecha_ingreso":       ss(row.get("joined")),
        "expiracion_contrato": ss(row.get("contract_expires")),
        "valor_actual_eur":    sf(row.get(valor_col)),
        "valor_maximo_eur":    sf(row.get("value_max")),
        "crecimiento_valor_x": round(float(row["value_growth_pct"]), 2) if pd.notna(row.get("value_growth_pct")) else None,
        "n_valoraciones":      si(row.get("n_valuations")),
        "goles_carrera":       si(row.get("total_goals")),
        "asistencias_carrera": si(row.get("total_assists")),
        "minutos_carrera":     si(row.get("total_minutes")),
        "contribucion_goles":  si(row.get("goal_contrib")),
        "temporadas_activas":  si(row.get("n_seasons_active")),
        "competiciones":       si(row.get("n_competitions")),
        "tarjetas_amarillas":  si(row.get("total_yellow")),
        "tarjetas_rojas":      si(row.get("total_red")),
        "partidos_internacionales": si(row.get("intl_matches")),
        "goles_internacionales":    si(row.get("intl_goals")),
        "seleccion_activa":         bool(row.get("is_current_national", False)),
        "n_lesiones":    si(row.get("n_injuries")),
        "dias_baja":     si(row.get("total_days_missed")),
        "partidos_baja": si(row.get("total_games_missed")),
        "n_transferencias": si(row.get("n_transfers")),
        "fee_maximo_eur":   sf(row.get("max_fee")) or 0,
    }

    # Métricas derivadas
    ns = max(ctx["temporadas_activas"], 1)
    ctx["goles_por_temporada"]       = round(ctx["goles_carrera"] / ns, 1)
    ctx["asistencias_por_temporada"] = round(ctx["asistencias_carrera"] / ns, 1)
    ctx["minutos_por_temporada"]     = round(ctx["minutos_carrera"] / ns, 0)
    if ctx["goles_carrera"] > 0 and ctx["minutos_carrera"] > 0:
        ctx["minutos_por_gol"] = round(ctx["minutos_carrera"] / ctx["goles_carrera"], 0)

    # Proyecciones Montecarlo
    if df_sim is not None and "player_id" in df_sim.columns:
        sim_row = df_sim[df_sim["player_id"] == player_id]
        if not sim_row.empty:
            sim_row = sim_row.iloc[0]
            ctx["proyeccion"] = {}
            for h in [1, 3, 5]:
                ctx["proyeccion"][f"{h}_anos"] = {
                    k: round(float(sim_row[f"h{h}_{k}"]) / 1e6, 1)
                    for k in ["pesimista","base","optimista"]
                    if f"h{h}_{k}" in sim_row.index and pd.notna(sim_row[f"h{h}_{k}"])
                }
            if "upside_3y_pct" in sim_row.index:
                ctx["upside_3y_pct"] = round(float(sim_row["upside_3y_pct"]), 1)

    return ctx


# Test
top_id   = df_base[df_base[valor_col].notna()].nlargest(1, valor_col)["player_id"].iloc[0]
ctx_test = construir_contexto_jugador(top_id)
print(f"✓ Contexto construido para: {ctx_test['nombre']}")
print(f"  Campos: {len(ctx_test)} | Proyecciones: {'Sí' if 'proyeccion' in ctx_test else 'No'}")

✓ Contexto construido para: Lamine Yamal
  Campos: 40 | Proyecciones: Sí


## Análisis — Cómo evaluar la calidad de los reportes

Una vez ejecutadas las celdas 6, 7 y 8, verifica lo siguiente:

**1. Coherencia de datos**
Los números en el reporte deben coincidir exactamente con los del dataset.
Si el reporte dice "23 goles en la temporada" pero el jugador tiene 23 goles
en toda su carrera, hay una interpretación incorrecta del prompt.
Solución: hacer el prompt más explícito indicando "goles TOTALES en toda la carrera".

**2. Sección "Señales de alerta"**
Es la más útil para un reclutador. Verifica que identifique correctamente:
- Contratos que expiran pronto → urgencia de fichaje
- Historial de lesiones elevado → riesgo médico real
- Valor estancado o en caída → señal de que el mercado perdió confianza
- Pocos minutos para la edad → falta de oportunidades o rendimiento bajo

**3. Sección "Ventana de oportunidad"**
Debe combinar la fecha de expiración del contrato con la edad del jugador.
Un jugador de 21 años con contrato hasta 2026 tiene una ventana diferente
a uno de 25 con contrato que vence en 3 meses.

**4. Modo interactivo como validación**
Prueba `analizar_jugador('nombre')` con jugadores que conozcas bien
del fútbol real. Es la mejor forma de detectar si el sistema genera
análisis coherentes con la realidad del jugador.

In [8]:
# Sistema de prompts
# ===========================================================

SYSTEM_PROMPT = """Eres un analista experto en scouting de fútbol con 15 años de experiencia
evaluando jóvenes talentos para clubes de primer nivel europeo.

Reglas estrictas:
- Usa SOLO los datos proporcionados, nunca inventes estadísticas
- Si un dato es N/D, indícalo sin suponer nada
- Valores monetarios siempre en millones de euros (M€)
- Responde siempre en español
- Sé directo y conciso, sin frases de relleno
- Contextualiza las estadísticas según la liga, edad y posición del jugador"""


def prompt_reporte_individual(ctx: dict) -> str:
    val     = f"{ctx['valor_actual_eur']/1e6:.1f}M€" if ctx.get("valor_actual_eur") else "No disponible"
    val_max = f"{ctx['valor_maximo_eur']/1e6:.1f}M€" if ctx.get("valor_maximo_eur") else "N/D"
    fee     = f"{ctx['fee_maximo_eur']/1e6:.1f}M€"   if ctx.get("fee_maximo_eur", 0) > 0 else "Sin fee registrado"

    proy_texto = ""
    if ctx.get("proyeccion"):
        proy_texto = "\nPROYECCIONES MONTECARLO (10,000 simulaciones):"
        for h_key, h_data in ctx["proyeccion"].items():
            anos = h_key.replace("_anos","")
            proy_texto += (
                f"\n  +{anos} años → "
                f"Pesimista: {h_data.get('pesimista','N/D')}M€  "
                f"Base: {h_data.get('base','N/D')}M€  "
                f"Optimista: {h_data.get('optimista','N/D')}M€"
            )
        if ctx.get("upside_3y_pct"):
            proy_texto += f"\n  Upside base a 3 años: +{ctx['upside_3y_pct']}%"

    return f"""Genera un reporte de scouting profesional con los siguientes datos.

DATOS DEL JUGADOR:
Nombre: {ctx['nombre']} | Edad: {ctx['edad']} años
Posición: {ctx['posicion_det']} ({ctx['posicion']})
Nacionalidad: {ctx['nacionalidad']} | País: {ctx['pais']}
Pie: {ctx['pie']} | Altura: {ctx.get('altura_cm','N/D')}cm | UE: {'Sí' if ctx['es_eu'] else 'No'}

CLUB Y CONTRATO:
Club: {ctx['club_actual']} — {ctx['liga']} ({ctx['pais_club']})
Ingresó: {ctx['fecha_ingreso']} | Contrato hasta: {ctx['expiracion_contrato']}

VALOR DE MERCADO:
Actual: {val} | Máximo histórico: {val_max}
Crecimiento total: {ctx.get('crecimiento_valor_x','N/D')}x | Fee máximo: {fee}
{proy_texto}

RENDIMIENTO:
Goles: {ctx['goles_carrera']} ({ctx['goles_por_temporada']}/temp) | Asistencias: {ctx['asistencias_carrera']} ({ctx['asistencias_por_temporada']}/temp)
Minutos: {ctx['minutos_carrera']:,} ({ctx['minutos_por_temporada']:.0f}/temp) | Temporadas: {ctx['temporadas_activas']} | Competiciones: {ctx['competiciones']}
Amarillas: {ctx['tarjetas_amarillas']} | Rojas: {ctx['tarjetas_rojas']}

SELECCIÓN NACIONAL:
Partidos: {ctx['partidos_internacionales']} | Goles: {ctx['goles_internacionales']} | Convocado actualmente: {'Sí' if ctx['seleccion_activa'] else 'No'}

HISTORIAL MÉDICO:
Lesiones: {ctx['n_lesiones']} | Días de baja: {ctx['dias_baja']} | Partidos perdidos: {ctx['partidos_baja']}

TRANSFERENCIAS:
Movimientos: {ctx['n_transferencias']} | Fee máximo: {fee}

Responde con EXACTAMENTE estas secciones:

## PERFIL EJECUTIVO
(2-3 oraciones resumiendo quién es y por qué es relevante)

## FORTALEZAS
(3-4 puntos concretos con los números que los respaldan)

## SEÑALES DE ALERTA
(2-3 riesgos identificados en los datos. Si no hay señales preocupantes, indicarlo)

## PROYECCIÓN DE VALOR
(Análisis del potencial económico con los datos disponibles)

## VENTANA DE OPORTUNIDAD
(Urgencia: contrato, edad, momento de carrera)

## RECOMENDACIÓN
(ALTA PRIORIDAD / SEGUIMIENTO / DESCARTAR — justificación de 2-3 líneas)"""


def prompt_comparativa(contextos: list) -> str:
    jugadores_texto = ""
    for i, ctx in enumerate(contextos, 1):
        val  = f"{ctx['valor_actual_eur']/1e6:.1f}M€" if ctx.get("valor_actual_eur") else "N/D"
        proy = ""
        if ctx.get("proyeccion") and "3_anos" in ctx["proyeccion"]:
            base = ctx["proyeccion"]["3_anos"].get("base","N/D")
            proy = f" → {base}M€ en 3 años"
        jugadores_texto += f"""
  {i}. {ctx['nombre']} | {ctx['edad']}a | {ctx['posicion_det']}
     Club: {ctx['club_actual']} ({ctx['liga']}) | Valor: {val}{proy}
     Goles/temp: {ctx['goles_por_temporada']} | Asist/temp: {ctx['asistencias_por_temporada']}
     Internacionales: {ctx['partidos_internacionales']} | Lesiones: {ctx['n_lesiones']} ({ctx['dias_baja']} días)
     Contrato hasta: {ctx['expiracion_contrato']}"""

    return f"""Compara estos {len(contextos)} jugadores para un club europeo con presupuesto de 5M€–30M€.

CANDIDATOS:{jugadores_texto}

Responde con:

## TABLA COMPARATIVA
(Tabla con métricas clave)

## ANÁLISIS COMPARATIVO
(2-3 párrafos: rendimiento, valor, riesgo, proyección)

## RANKING DE PRIORIDAD
(Ordenados con justificación de 2 líneas cada uno)

## RECOMENDACIÓN FINAL
(El jugador que priorizarías y por qué en 3-4 líneas)"""


def prompt_joyas_ocultas(joyas_df: pd.DataFrame, n: int = 5) -> str:
    top = joyas_df.head(n)
    texto = ""
    for _, row in top.iterrows():
        nombre = str(row.get("nombre", row.get("player_name","N/A"))).split("(")[0].strip()
        val    = f"{row.get('valor_actual', row.get('latest_value', 0))/1e6:.2f}M€"
        proy   = f"{row.get('h3_base',0)/1e6:.1f}M€" if row.get("h3_base") else "N/D"
        upside = f"{row.get('upside_3y_pct',0):.0f}%"
        edad   = row.get("edad_actual", row.get("age","N/A"))
        pos    = row.get("posicion", row.get("main_position","N/A"))
        texto += f"\n  - {nombre} | {edad}a | {pos} | Valor: {val} | Proyección 3a: {proy} | Upside: {upside}"

    return f"""Analiza estos {n} jugadores con alto potencial que el mercado aún no valora completamente.

CANDIDATOS:{texto}

Para cada uno: por qué es una oportunidad, cuál es el riesgo principal, en qué ventana actuar.
Termina con una conclusión sobre la mejor oportunidad."""


print("✓ Sistema de prompts definido")

✓ Sistema de prompts definido


In [9]:
# Función de llamada al LLM
# ===========================================================

def llamar_llm(
    prompt: str,
    system: str = SYSTEM_PROMPT,
    max_tokens: int = 1500,
    temperatura: float = 0.3,
) -> str:
    """Llama a Groq y retorna la respuesta como texto."""
    try:
        response = client.chat.completions.create(
            model=MODELO,
            max_tokens=max_tokens,
            temperature=temperatura,
            messages=[
                {"role": "system", "content": system},
                {"role": "user",   "content": prompt},
            ],
        )
        return response.choices[0].message.content

    except Exception as e:
        if "rate_limit" in str(e).lower() or "429" in str(e):
            print("  ⚠ Rate limit — esperando 30s...")
            time.sleep(30)
            return llamar_llm(prompt, system, max_tokens, temperatura)
        print(f"  ✗ Error: {e}")
        return f"Error: {str(e)}"


def guardar_reporte(texto: str, nombre_archivo: str) -> Path:
    path = OUTPUT_DIR / f"{nombre_archivo}.md"
    with open(path, "w", encoding="utf-8") as f:
        f.write(texto)
    print(f"  ✓ {path}")
    return path


# Test de conexión
print("Verificando conexión con Groq...")
test = llamar_llm("Responde solo: OK", max_tokens=10)
print(f"  Respuesta: {test.strip()}")

Verificando conexión con Groq...
  Respuesta: OK


In [10]:
# Reportes individuales Top 5
# ===========================================================

print(f"\nGenerando reportes Top 5...\n")

top5_ids = df_base[df_base[valor_col].notna()].nlargest(5, valor_col)["player_id"].tolist()
reportes_individuales = {}

for i, pid in enumerate(top5_ids, 1):
    ctx    = construir_contexto_jugador(pid)
    nombre = ctx["nombre"]
    print(f"  [{i}/5] {nombre}...")

    reporte = llamar_llm(prompt_reporte_individual(ctx))
    reportes_individuales[pid] = {"nombre": nombre, "contexto": ctx, "reporte": reporte}
    guardar_reporte(reporte, f"reporte_{nombre.replace(' ','_').lower()}")

    print(f"\n{'─'*60}\n  {nombre}\n{'─'*60}")
    print(reporte)
    print()

    if i < len(top5_ids):
        time.sleep(3)

print(f"✓ {len(reportes_individuales)} reportes generados")


Generando reportes Top 5...

  [1/5] Lamine Yamal...
  ✓ ..\outputs\paso4\reporte_lamine_yamal.md

────────────────────────────────────────────────────────────
  Lamine Yamal
────────────────────────────────────────────────────────────
## PERFIL EJECUTIVO
Lamine Yamal es un joven extremo derecho de 17.5 años con un gran potencial, jugando para el FC Barcelona en LaLiga. Su valor de mercado actual es de 180.0M€ y ha demostrado un rendimiento destacado con 27 goles y 37 asistencias en 4 temporadas.

## FORTALEZAS
* Rendimiento goleador: 27 goles en 4 temporadas, con un promedio de 6.8 goles por temporada.
* Capacidad asistencial: 37 asistencias en 4 temporadas, con un promedio de 9.2 asistencias por temporada.
* Participación en la selección nacional: 44 partidos y 18 goles, lo que demuestra su nivel y reconocimiento a nivel internacional.
* Minutos jugados: 1,588 minutos en 4 temporadas, con un promedio de 397 minutos por temporada.

## SEÑALES DE ALERTA
* Lesiones: 6 lesiones en su ca

In [11]:
# Comparativas por posición
# ===========================================================

print("\nGenerando comparativas...\n")
comparativas = {}

for posicion in ["Attack", "Midfield", "Defender"]:
    ids_pos = (
        df_base[(df_base[valor_col].notna()) & (df_base["main_position"] == posicion)]
        .nlargest(3, valor_col)["player_id"].tolist()
    )
    if len(ids_pos) < 2:
        continue

    contextos = [construir_contexto_jugador(pid) for pid in ids_pos]
    nombres   = " vs ".join(c["nombre"] for c in contextos)
    print(f"  {posicion}: {nombres}")

    comp = llamar_llm(prompt_comparativa(contextos))
    comparativas[posicion] = comp
    guardar_reporte(comp, f"comparativa_{posicion.lower()}")

    print(f"\n{'─'*60}\n  COMPARATIVA: {posicion}\n{'─'*60}")
    print(comp)
    print()
    time.sleep(3)

print(f"✓ {len(comparativas)} comparativas generadas")


Generando comparativas...

  Attack: Lamine Yamal vs Erling Haaland vs Vinicius Junior
  ✓ ..\outputs\paso4\comparativa_attack.md

────────────────────────────────────────────────────────────
  COMPARATIVA: Attack
────────────────────────────────────────────────────────────
## TABLA COMPARATIVA
| Jugador | Edad | Posición | Valor Actual (M€) | Valor en 3 años (M€) | Goles/temp | Asist/temp | Internacionales | Lesiones (días) |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| Lamine Yamal | 17.5 | Right Winger | 180.0 | 418.8 | 6.8 | 9.2 | 44 | 6 (96) |
| Erling Haaland | 24.4 | Centre-Forward | 176.0 | 174.6 | 20.2 | 4.4 | 98 | 15 (277) |
| Vinicius Junior | 24.5 | Left Winger | 168.0 | 167.8 | 9.7 | 7.2 | 70 | 9 (191) |

## ANÁLISIS COMPARATIVO
El análisis de estos jugadores muestra diferencias significativas en su valor, rendimiento y riesgo. Lamine Yamal, a pesar de su juventud, presenta un valor en constante aumento y un rendimiento notable en goles y asistencias por temp

In [12]:
# Joyas ocultas
# ===========================================================

print("\nGenerando análisis de joyas ocultas...\n")

if df_joyas is not None and len(df_joyas) >= 3:
    joyas_input = df_joyas
else:
    print("  Construyendo joyas desde el dataset base...")
    threshold    = df_base[valor_col].quantile(0.5)
    joyas_input  = (
        df_base[
            df_base[valor_col].notna() &
            (df_base[valor_col] <= threshold) &
            (df_base["intl_matches"] > 5)
        ]
        .nlargest(10, "intl_matches")
        .rename(columns={
            "nombre_limpio": "nombre",
            "age":           "edad_actual",
            "main_position": "posicion",
            valor_col:       "valor_actual",
        })
    )

reporte_joyas = llamar_llm(prompt_joyas_ocultas(joyas_input, n=5))
guardar_reporte(reporte_joyas, "joyas_ocultas_analisis")
print(f"{'─'*60}\n  JOYAS OCULTAS\n{'─'*60}")
print(reporte_joyas)


Generando análisis de joyas ocultas...
  ✓ ..\outputs\paso4\joyas_ocultas_analisis.md
────────────────────────────────────────────────────────────
  JOYAS OCULTAS
────────────────────────────────────────────────────────────
**Análisis de los 5 jugadores**

1. **Geovany Quenda**: Oportunidad por su alto upside (134%) y proyección de valor en 3 años (107.3M€). Riesgo principal: su juventud (17.7 años) y la incertidumbre sobre su adaptación a un nivel superior. Ventana de actuación: inmediata, aprovechando su valor actual (45.93M€) antes de que aumente.
2. **Estêvão**: Oportunidad similar a Quenda, con un upside de 133% y proyección de valor en 3 años (120.2M€). Riesgo principal: su valor actual (51.65M€) es más alto que el de Quenda, lo que puede aumentar el riesgo de no alcanzar la proyección. Ventana de actuación: inmediata, antes de que su valor aumente aún más.
3. **Ethan Nwaneri**: Oportunidad por su posición en el mediocampo y su proyección de valor en 3 años (119.7M€). Riesgo pri

## Limitaciones importantes

**El LLM no tiene conocimiento actualizado del jugador.**
Solo sabe lo que le pasamos en el prompt. Si un jugador tuvo una lesión
grave este mes, el sistema no lo sabe a menos que esté en los datos.

**Posibles alucinaciones en el contexto.**
Si el modelo inventa información que no está en el prompt, hay que
reforzar el system prompt con: "Si no tienes el dato, di explícitamente
que no está disponible. Nunca inventes estadísticas."

**Los valores de proyección son probabilísticos.**
El LLM los presenta como rangos (Montecarlo), no como predicciones exactas.
Es importante que el reclutador entienda esto — el "escenario base" no es
una garantía sino la mediana de 10,000 simulaciones.

In [13]:
# Modo interactivo
# ===========================================================

def buscar_jugador(nombre_parcial: str) -> pd.DataFrame:
    """Busca jugadores por nombre parcial."""
    mask = df_base["nombre_limpio"].str.contains(nombre_parcial, case=False, na=False)
    return df_base[mask][
        ["player_id","nombre_limpio","age","main_position","current_club_name",valor_col]
    ].head(10)


def analizar_jugador(nombre_o_id, mostrar_contexto: bool = False):
    """
    Genera un reporte para cualquier jugador del dataset.

    Uso:
        analizar_jugador("Pedri")
        analizar_jugador("Bellingham")
        analizar_jugador(683840)              ← player_id directo
        analizar_jugador("Yamal", mostrar_contexto=True)
    """
    if isinstance(nombre_o_id, str):
        resultados = buscar_jugador(nombre_o_id)
        if resultados.empty:
            print(f"  No se encontró '{nombre_o_id}'")
            return
        if len(resultados) > 1:
            print(f"  {len(resultados)} coincidencias encontradas:")
            print(resultados[["player_id","nombre_limpio","age",
                               "main_position","current_club_name"]].to_string(index=False))
            print("\n  Usando la primera. Para otra, pasa el player_id directamente.")
        pid = int(resultados.iloc[0]["player_id"])
    else:
        pid = int(nombre_o_id)

    ctx     = construir_contexto_jugador(pid)
    reporte = llamar_llm(prompt_reporte_individual(ctx))

    if mostrar_contexto:
        print("  CONTEXTO ENVIADO AL LLM:")
        for k, v in ctx.items():
            if k != "proyeccion":
                print(f"    {k}: {v}")
        print()

    guardar_reporte(reporte, f"consulta_{ctx['nombre'].replace(' ','_').lower()}")

    print(f"\n{'═'*60}\n  REPORTE: {ctx['nombre']}\n{'═'*60}")
    print(reporte)
    return reporte


print("✓ Modo interactivo listo")
print()


# ── Descomenta para probar:
analizar_jugador("Myles Lewis-Skelly")

✓ Modo interactivo listo
  ✓ ..\outputs\paso4\consulta_myles_lewis-skelly.md

════════════════════════════════════════════════════════════
  REPORTE: Myles Lewis-Skelly
════════════════════════════════════════════════════════════
## PERFIL EJECUTIVO
Myles Lewis-Skelly es un joven defensor lateral izquierdo de 18.3 años que juega para el Arsenal FC en la Premier League. Con un valor de mercado actual de 44.5M€ y un crecimiento total de 21.5x, es un jugador relevante en el mercado. Su participación en la selección nacional y su rendimiento en el club lo convierten en un prospecto interesante.

## FORTALEZAS
* Rendimiento ofensivo destacado, con 6 goles y 8 asistencias en 6 temporadas, lo que indica una capacidad para contribuir en el ataque.
* Participación regular en la selección nacional, con 39 partidos jugados y 3 goles anotados, lo que sugiere un nivel de confianza en su capacidad.
* Minutos jugados significativos, con 3,539 minutos en 6 temporadas, lo que indica una capacidad para 

'## PERFIL EJECUTIVO\nMyles Lewis-Skelly es un joven defensor lateral izquierdo de 18.3 años que juega para el Arsenal FC en la Premier League. Con un valor de mercado actual de 44.5M€ y un crecimiento total de 21.5x, es un jugador relevante en el mercado. Su participación en la selección nacional y su rendimiento en el club lo convierten en un prospecto interesante.\n\n## FORTALEZAS\n* Rendimiento ofensivo destacado, con 6 goles y 8 asistencias en 6 temporadas, lo que indica una capacidad para contribuir en el ataque.\n* Participación regular en la selección nacional, con 39 partidos jugados y 3 goles anotados, lo que sugiere un nivel de confianza en su capacidad.\n* Minutos jugados significativos, con 3,539 minutos en 6 temporadas, lo que indica una capacidad para mantener un nivel de rendimiento constante.\n* Altura y pie izquierdo, lo que puede ser una ventaja en la posición de defensor lateral izquierdo.\n\n## SEÑALES DE ALERTA\n* Lesiones: aunque solo ha tenido 1 lesión, es impor

In [14]:
# Compilar informe completo
# ===========================================================

def compilar_informe(reportes: dict, comparativas: dict, joyas: str = None) -> Path:
    lineas = [
        "# Informe de Scouting — Jóvenes Promesas de Fútbol",
        "### Llama 3.3 70B (Groq) · Datos Transfermarkt 2003–2025",
        "", "---", "",
        "## SECCIÓN 1 — Reportes Individuales", "",
    ]
    for data in reportes.values():
        lineas += [f"### {data['nombre']}", "", data["reporte"], "", "---", ""]

    lineas += ["## SECCIÓN 2 — Comparativas por Posición", ""]
    for pos, comp in comparativas.items():
        lineas += [f"### {pos}", "", comp, "", "---", ""]

    if joyas:
        lineas += ["## SECCIÓN 3 — Joyas Ocultas", "", joyas]

    path = OUTPUT_DIR / "informe_completo_scouting.md"
    path.write_text("\n".join(lineas), encoding="utf-8")
    print(f"✓ Informe completo: {path}")
    return path


try:
    rj = reporte_joyas if "reporte_joyas" in dir() else None
    compilar_informe(reportes_individuales, comparativas, rj)
except Exception as e:
    print(f"⚠ {e}")

✓ Informe completo: ..\outputs\paso4\informe_completo_scouting.md


In [16]:
# Resumen final
# ===========================================================

print("\n" + "═"*62)
print("  RESUMEN PASO 4")
print("═"*62)
print(f"""
  LLM:                  Llama 3.3 70B vía Groq (gratuito)
  Reportes individuales: {len(reportes_individuales)}
  Comparativas:          {len(comparativas)}

  Outputs en outputs/paso4/:
    reporte_[jugador].md
    comparativa_[posicion].md
    joyas_ocultas_analisis.md
    informe_completo_scouting.md

  Modo interactivo:
    analizar_jugador('nombre')
""")


══════════════════════════════════════════════════════════════
  RESUMEN PASO 4
══════════════════════════════════════════════════════════════

  LLM:                  Llama 3.3 70B vía Groq (gratuito)
  Reportes individuales: 5
  Comparativas:          3

  Outputs en outputs/paso4/:
    reporte_[jugador].md
    comparativa_[posicion].md
    joyas_ocultas_analisis.md
    informe_completo_scouting.md

  Modo interactivo:
    analizar_jugador('nombre')
